# L6-3b Demo: Reward shaping can fail

This notebook keeps the algorithm fixed and changes only the reward:
- `sparse_1m`: a hard 1-meter success gate
- `dense_vel`: the direct forward-velocity signal
- `shaped`: forward velocity minus an energy penalty

The pedagogical point is the Lecture 3 reward-design story: sparse rewards make credit assignment hard, but even "helpful" shaping can pull the policy toward the wrong behavior if the proxy is easier to optimize than the real objective.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from l6_3_utils import (
    CMM_AMBER,
    CMM_MOSS,
    CMM_ROSE,
    CMM_SLATE,
    FIGURE_DIR,
    rollout_policy,
    save_figure,
    smooth_xy,
    train_or_load,
)

TOTAL_TIMESTEPS = 50_000
SEED = 11

reward_envs = {
    "sparse_1m": dict(include_velocity=True, action_mode="torque", reward_mode="sparse_1m"),
    "dense_vel": dict(include_velocity=True, action_mode="torque", reward_mode="dense_vel"),
    "shaped": dict(
        include_velocity=True,
        action_mode="torque",
        reward_mode="shaped",
        shaping_coef=0.05,
    ),
}

print(f"Saving lecture figures into: {FIGURE_DIR}")


In [ ]:
sparse_artifact = train_or_load(
    "ppo",
    "l6_3b_ppo_sparse.zip",
    reward_envs["sparse_1m"],
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
)
dense_artifact = train_or_load(
    "ppo",
    "l6_3b_ppo_dense.zip",
    reward_envs["dense_vel"],
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
)
shaped_artifact = train_or_load(
    "ppo",
    "l6_3b_ppo_shaped.zip",
    reward_envs["shaped"],
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
)

print("Recent sparse reward:", np.mean(sparse_artifact.rewards[-5:]))
print("Recent dense reward:", np.mean(dense_artifact.rewards[-5:]))
print("Recent shaped reward:", np.mean(shaped_artifact.rewards[-5:]))


In [ ]:
sparse_rollout = rollout_policy(sparse_artifact.model, reward_envs["sparse_1m"], seed=SEED)
dense_rollout = rollout_policy(dense_artifact.model, reward_envs["dense_vel"], seed=SEED)
shaped_rollout = rollout_policy(shaped_artifact.model, reward_envs["shaped"], seed=SEED)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))

for label, artifact, color in [
    ("dense", dense_artifact, CMM_MOSS),
    ("sparse", sparse_artifact, CMM_ROSE),
    ("shaped", shaped_artifact, CMM_AMBER),
]:
    x, y = smooth_xy(artifact.steps, artifact.rewards, window=5)
    axes[0].plot(x, y, lw=2.0, color=color, label=label)

axes[0].set_xlabel("environment steps")
axes[0].set_ylabel("episode reward")
axes[0].set_title("Training curves", color=CMM_SLATE)
axes[0].grid(True, alpha=0.25)
axes[0].legend(loc="lower right", fontsize=8)

axes[1].plot(dense_rollout["x"], color=CMM_MOSS, lw=2.0, label="dense")
axes[1].plot(sparse_rollout["x"], color=CMM_ROSE, lw=2.0, label="sparse")
axes[1].plot(shaped_rollout["x"], color=CMM_AMBER, lw=2.0, label="shaped")
axes[1].set_xlabel("rollout step")
axes[1].set_ylabel("torso x-position")
axes[1].set_title("True x-displacement @ eval", color=CMM_SLATE)
axes[1].grid(True, alpha=0.25)
axes[1].legend(loc="upper left", fontsize=8)

fig_path = save_figure(fig, "reward_shaping_fail.pdf")
print("Final dense x:", float(dense_rollout["x"][-1]))
print("Final sparse x:", float(sparse_rollout["x"][-1]))
print("Final shaped x:", float(shaped_rollout["x"][-1]))
fig_path


The Assignment 3 contrast is deliberate: `Assignments/cmm-26-a3/animRL/rewards/rewards.py` uses a product-style reward so failures in one requirement cannot be compensated by over-optimizing another term. The crawler examples here are additive signals, so the agent can trade one term against another much more freely.
